<a href="https://colab.research.google.com/github/columnadominic-profile/freeCodeCamp.org-boilerplate-neural-network-sms-text-classifier-Solution/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

print("TensorFlow successfully loaded!")
print("Version:", tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# 1. Load TSV datasets into Pandas DataFrames
train_df = pd.read_csv(train_file_path, sep='\t', names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', names=['label', 'message'])

# 2. Encode categorical labels (ham: 0, spam: 1)
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['label'].map({'ham': 0, 'spam': 1})

train_labels = train_df.pop('label')
test_labels = test_df.pop('label')

# 3. Create TextVectorization layer to convert words into integer sequences
VOCAB_SIZE = 1000
MAX_SEQUENCE_LENGTH = 100

encoder = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH
)
encoder.adapt(train_df['message'].values)

# 4. Build the Sequential Neural Network Model
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=64,
        mask_zero=True
    ),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 5. Compile the model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# 6. Train the model
model.fit(
    train_df['message'].values,
    train_labels.values,
    epochs=10,
    validation_data=(test_df['message'].values, test_labels.values),
    validation_steps=30,
    verbose=1
)

In [ ]:
# Function to predict messages based on the trained model
def predict_message(pred_text):
    # 1. Convert the input string into a TensorFlow string tensor for Keras compatibility
    input_data = tf.constant([pred_text], dtype=tf.string)

    # 2. Predict the probability (0 = ham, 1 = spam) using the trained model
    prob = float(model.predict(input_data, verbose=0)[0][0])

    # 3. Assign 'spam' if probability is 0.5 or greater, otherwise 'ham'
    label = 'spam' if prob >= 0.5 else 'ham'

    return [prob, label]

# Test the function with a sample message
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
